<a href="https://colab.research.google.com/github/zaidlameer/DeetectorPrototype/blob/main/knowledgeDistilltionV3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# prompt: mount drive

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from transformers import ViTForImageClassification, AutoModelForImageClassification
import torch
import torch.nn as nn

# Load teacher model (pretrained on deepfake detection)
teacher_model = ViTForImageClassification.from_pretrained("prithivMLmods/Deep-Fake-Detector-Model").to("cuda")

# Freeze teacher model parameters
for param in teacher_model.parameters():
    param.requires_grad = False

# # Alternative 1: Use a smaller ViT model as student
# student_model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224").to("cuda")
# # Modify the classifier for 2 classes (real/fake)
# student_model.classifier = nn.Linear(student_model.classifier.in_features, 2).to("cuda")

# Alternative 2: Use DeiT (Data-efficient image Transformer)
student_model = AutoModelForImageClassification.from_pretrained("facebook/deit-tiny-patch16-224").to("cuda")
student_model.classifier = nn.Linear(student_model.classifier.in_features, 2).to("cuda")

# Alternative 3: Use a MobileViT (lightweight mobile-friendly ViT)
# student_model = AutoModelForImageClassification.from_pretrained("apple/mobilevit-xx-small").to("cuda")
# student_model.classifier = nn.Linear(student_model.classifier.in_features, 2).to("cuda")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.6k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/23.0M [00:00<?, ?B/s]

In [3]:
!pip install datasets

model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [4]:
import torch
from torchvision.models import resnet50
from torch.utils.data import DataLoader
from transformers import ViTImageProcessor
from datasets import load_from_disk

# Load preprocessed dataset from Drive
dataset_path = '/content/drive/My Drive/preprocessed_dataset'
dataset = load_from_disk(dataset_path)

Loading dataset from disk:   0%|          | 0/85 [00:00<?, ?it/s]

In [5]:
import torch.nn.functional as F

def distillation_loss(student_logits, teacher_logits, true_labels, temperature=2.0, alpha=0.5):
    """Knowledge Distillation Loss (KL Divergence + Cross-Entropy)."""
    # Soft target loss (KL Divergence)
    kl_loss = F.kl_div(
        F.log_softmax(student_logits / temperature, dim=1),
        F.softmax(teacher_logits / temperature, dim=1),
        reduction="batchmean"
    ) * (temperature ** 2)

    # Hard target loss (Cross-Entropy)
    ce_loss = F.cross_entropy(student_logits, true_labels)

    # Combine losses
    return alpha * ce_loss + (1 - alpha) * kl_loss


In [6]:
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
from tqdm import tqdm
import torch
import os

# Create DataLoaders from your dataset
train_size = int(0.8 * len(dataset["train"]))
test_size = len(dataset["train"]) - train_size
train_dataset, test_dataset = random_split(dataset["train"], [train_size, test_size])

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
eval_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Optimizer
optimizer = optim.AdamW(student_model.parameters(), lr=5e-5)

# Checkpoint directory
checkpoint_dir = "content/drive/MyDrive/kdv3/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# Load checkpoint if exists
start_epoch = 0
checkpoint_path = os.path.join(checkpoint_dir, "latest_checkpoint.pth")
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    student_model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    print(f"Loaded checkpoint from epoch {start_epoch}")

# Training loop
num_epochs = 5

for epoch in range(start_epoch, num_epochs):
    student_model.train()
    total_loss = 0
    progress_bar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch + 1}")

    for batch_idx, batch in progress_bar:
        inputs, labels = batch["pixel_values"].to("cuda"), batch["labels"].to("cuda")

        with torch.no_grad():
            teacher_logits = teacher_model(inputs).logits

        student_logits = student_model(inputs).logits
        loss = distillation_loss(student_logits, teacher_logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({"loss": total_loss / (batch_idx + 1)})

    # Save checkpoint at the end of each epoch
    torch.save({
        "epoch": epoch,
        "model_state_dict": student_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, checkpoint_path)

    print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_dataloader)}")
    print(f"Checkpoint saved at epoch {epoch + 1}")


Epoch 1: 100%|██████████| 1750/1750 [40:20<00:00,  1.38s/it, loss=0.197]


Epoch 1, Loss: 0.19695365542173385
Checkpoint saved at epoch 1


Epoch 2: 100%|██████████| 1750/1750 [30:24<00:00,  1.04s/it, loss=0.178]


Epoch 2, Loss: 0.17798921459487507
Checkpoint saved at epoch 2


Epoch 3: 100%|██████████| 1750/1750 [30:51<00:00,  1.06s/it, loss=0.174]


Epoch 3, Loss: 0.1744055637248925
Checkpoint saved at epoch 3


Epoch 4: 100%|██████████| 1750/1750 [30:38<00:00,  1.05s/it, loss=0.172]


Epoch 4, Loss: 0.17164877784252167
Checkpoint saved at epoch 4


Epoch 5: 100%|██████████| 1750/1750 [30:39<00:00,  1.05s/it, loss=0.17]


Epoch 5, Loss: 0.17030789489405496
Checkpoint saved at epoch 5


In [7]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

student_model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in eval_dataloader:
        inputs, labels = batch["pixel_values"].to("cuda"), batch["labels"].to("cuda")
        student_logits = student_model(inputs).logits
        predictions = torch.argmax(student_logits, dim=1).cpu().numpy()
        all_preds.extend(predictions)
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")

print(f"Student Model Accuracy: {accuracy:.4f}")
print(f"Student Model F1 Score: {f1:.4f}")

# Save the model
student_model.save_pretrained("distilled_vit_deepfake_model")


Student Model Accuracy: 0.9691
Student Model F1 Score: 0.9691


In [8]:
student_model.save_pretrained("content/drive/MyDrive/distilled_vit_deepfake_model_v3")